# 01 - Data Audit & Ingestion
## Renewable Energy Forecasting & Battery Storage Optimization

This notebook conducts a data audit of the Open Power System Data (OPSD) Time Series package (dated 2020-10-06).
It audits column structures, temporal continuity, missing value patterns across European bidding zones, and justifies the selection of **Denmark West (`DK_1`)** as our primary bidding zone.


In [1]:
import sys
sys.path.insert(0, '..')

from src.config import load_config
from src.data_loader import load_raw_country_data
from src.preprocessing import validate_and_clean_time_series, get_clean_data
import pandas as pd
import json

config = load_config()
print("Selected country:", config["data"]["selected_country"])
print("Configured columns:", config["data"]["columns_mapping"])


Selected country: DK_1
Configured columns: {'utc_timestamp': 'timestamp', 'DK_1_solar_generation_actual': 'solar_generation_mw', 'DK_1_wind_generation_actual': 'wind_generation_mw', 'DK_1_load_actual_entsoe_transparency': 'load_actual_mw', 'DK_1_load_forecast_entsoe_transparency': 'load_forecast_mw', 'DK_1_price_day_ahead': 'price_day_ahead_eur_mwh'}


### 1. Load Raw Country Observations & Verify Frequency

In [2]:
raw_df = load_raw_country_data(config=config, use_cache=True)
print(f"Total raw observations: {len(raw_df):,}")
print(f"Time span: {raw_df.index.min()} to {raw_df.index.max()}")
print("\nMissing values per column in raw OPSD data:")
print(raw_df.isna().sum())


[2026-10-25 14:02:37] [INFO] [src.data_loader]: Loading cached extracted data from /home/asus/Projects/Renewable-Energy-Intelligence/data/processed/DK_1_raw_extracted.parquet


Total raw observations: 50,401
Time span: 2014-12-31 23:00:00+00:00 to 2020-09-30 23:00:00+00:00

Missing values per column in raw OPSD data:
load_actual_mw              3
load_forecast_mw            3
price_day_ahead_eur_mwh    15
solar_generation_mw        12
wind_generation_mw          3
dtype: int64


### 2. Validate Regular Hourly Grid & Clean Series

In [3]:
clean_df, audit_summary = validate_and_clean_time_series(raw_df, config=config)
print("Data Quality & Cleaning Audit Summary:")
print(json.dumps(audit_summary, indent=2))
print("\nClean dataset summary statistics:")
display(clean_df.describe())


[2026-10-25 14:02:37] [INFO] [src.preprocessing]: Starting data preprocessing and quality audit...


[2026-10-25 14:02:37] [INFO] [src.preprocessing]: Applying time-interpolation for remaining isolated boundary gaps...


[2026-10-25 14:02:37] [INFO] [src.preprocessing]: Preprocessing complete. Clean dataset spans 2014-12-31 23:00:00+00:00 to 2020-09-30 23:00:00+00:00 (50,401 rows). Detected 13 flat telemetry segments.


Data Quality & Cleaning Audit Summary:
{
  "raw_total_rows": 50401,
  "raw_start": "2014-12-31 23:00:00+00:00",
  "raw_end": "2020-09-30 23:00:00+00:00",
  "duplicate_timestamps": 0,
  "missing_timestamps_on_grid": 0,
  "raw_missing_per_column": {
    "load_actual_mw": 3,
    "load_forecast_mw": 3,
    "price_day_ahead_eur_mwh": 15,
    "solar_generation_mw": 12,
    "wind_generation_mw": 3
  },
  "solar_negative_count": 0,
  "telemetry_gaps_count": 13,
  "telemetry_gaps": [
    {
      "start": "2015-08-25 16:00:00+00:00",
      "end": "2015-08-25 19:00:00+00:00",
      "hours": 4,
      "stuck_value_mw": 94.19
    },
    {
      "start": "2015-09-01 18:00:00+00:00",
      "end": "2015-09-02 06:00:00+00:00",
      "hours": 13,
      "stuck_value_mw": 9.33
    },
    {
      "start": "2015-12-30 14:00:00+00:00",
      "end": "2015-12-30 19:00:00+00:00",
      "hours": 6,
      "stuck_value_mw": 16.82
    },
    {
      "start": "2016-06-29 16:00:00+00:00",
      "end": "2016-06-29 18:0

,load_actual_mw,load_forecast_mw,price_day_ahead_eur_mwh,solar_generation_mw,wind_generation_mw,capacity_proxy_mw
count,50401.000000,50401.000000,50401.000000,50401.000000,50401.000000,50401.000000
mean,2273.601074,2274.465088,31.328835,69.042030,1293.385498,465.445129
std,450.992401,450.480255,14.851234,116.303139,957.556885,112.955185
min,1201.300049,1201.800049,-58.799999,0.000000,1.220000,161.710007
25%,1905.219971,1904.099976,22.639999,0.000000,469.029999,410.230011
50%,2234.669922,2242.100098,30.290001,2.180000,1099.829956,460.179993
75%,2647.909912,2648.000000,40.000000,93.940002,1990.380005,556.309998
max,6711.910156,3463.000000,200.039993,726.710022,7107.490234,726.710022


### 3. Verification of Physical Invariants
- Solar generation >= 0 MW
- Zero missing values on the complete UTC hourly index


In [4]:
assert (clean_df['solar_generation_mw'] >= 0).all(), "Negative solar generation detected!"
assert clean_df.isna().sum().sum() == 0, "Remaining NaNs detected!"
print("Physical sanity check passed successfully!")


Physical sanity check passed successfully!
